Community Assistanty to allow Backed by a Firebase Database

Business Justification: I have a community management solution built by me (https://audiatur.co). In simple terms, Audiatur provides all the tools(organising elections, conference management, event calendar, general info storage) needed to manage a community(a community could be a church, association etc).

One key requirement is to allow community members get info about the community using Whatsapp. This project makes that possible. Data is stored on Firebase

First the imports

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import firebase_admin
from firebase_admin import credentials, firestore
import json
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
import gradio as gr

Lets setup model

In [2]:

MODEL = "gpt-4.1-nano"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


now lets setup Firebase so we can load data

In [3]:
config = json.loads(os.getenv("FIREBASE_CONFIG_JSON"))
cred = credentials.Certificate(config)
try:
    firebase_admin.initialize_app(cred)
    print("Firebase initialized successfully!")
except ValueError:
    print("App already initialized.")



Firebase initialized successfully!


Load communities and conferences

In [4]:
db = firestore.client()
projects = db.collection('projects').get()
for project in projects:
    print(project.to_dict())


{'shortDescription': '', 'description': '', 'projectType': 'GAME', 'endDate': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc), 'id': 'f618e67b-baa0-42af-9386-bc4e0c63dd11', 'userId': '', 'createdAt': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc), 'status': 'INITIATED', 'createdById': '', 'startDate': DatetimeWithNanoseconds(2025, 9, 22, 19, 57, 47, 320000, tzinfo=datetime.timezone.utc), 'name': 'First Game'}
{'shortDescription': 'This is game 3, a game to test full implementation', 'endDate': DatetimeWithNanoseconds(2025, 11, 12, 5, 53, 7, 56000, tzinfo=datetime.timezone.utc), 'description': '', 'id': 'd4a6f319-3faf-44ee-aeb1-b3402f67d59f', 'questionSource': 'AIGENERATED', 'status': 'INITIATED', 'createdById': '', 'name': 'Game 3', 'domains': ['game3.audiatur.co'], 'projectType': 'GAME', 'userId': '', 'createdAt': DatetimeWithNanoseconds(2025, 11, 12, 5, 53, 7, 56000, tzinfo=datetime.timezone.utc), 'startDa

Lets create Langchain documents

In [ ]:
# def create_event_page_content(data):
#     return f"""
#     Event: {data.get('name', '') or ''}
#     Short Description: {data.get('shortDescription', '') or ''}
#     Description: {data.get('description', '') or ''}
#     Venue: {data.get('venue', '') or ''}
#     Start Date: {data.get('startDate', '') or ''}
#     End Date: {data.get('endDate', '') or ''}
#     Event Type: {data.get('projectType', '') or ''}
#     Status: {data.get('status', '') or ''}
#     Organisation Lookup id: {data.get('parentProjectId', '') or ''}    
#     """

def create_event_page_content(data):
    return {
        "name": data.get('name', ''),
        "shortDescription": data.get('shortDescription', ''),
        "description": data.get('description', ''),
        "venue": data.get('venue', ''),
        "startDate": data.get('startDate', ''),
        "endDate": data.get('endDate', ''),
        "projectType": data.get('projectType', ''),
        "status": data.get('status', ''),
        "parentProjectId": data.get('parentProjectId', '')   
    }


documents = [Document(
            page_content=create_event_page_content(project.to_dict()),
            metadata={'source': project.id})  
            for project in projects if project.to_dict().get('projectType') in ['COMMUNITY','CONFERENCE']]

for doc in documents:
    print(doc)

page_content='
    Event: WOW2026
    Short Description: Wonder of Worship 2026
    Description: WOW is an annual worship program hosted by the youth wing of the His Grace Assembly Area Parrish of the Redeem Christian Church of God
    Venue: RCCG: His Grace Assembly
    Start Date: 2026-08-31 23:00:00+00:00
    End Date: 2026-01-31 23:00:00+00:00
    Event Type: CONFERENCE
    Status: INITIATED
    Organisation Lookup id: k3EImfYWwxKQZ1UVI24v    
    ' metadata={'source': 'TfcHEfgSbFBSInTlMs6t'}
page_content='
    Event: SPENC
    Short Description: Society of Petroleum Engineers Nigeria Council
    Description: 
    Venue: 
    Start Date: 2026-03-11 05:40:00.876000+00:00
    End Date: 2026-03-11 05:40:00.876000+00:00
    Event Type: COMMUNITY
    Status: INITIATED
    Organisation Lookup id:     
    ' metadata={'source': 'Wu3W5E2T7QrPZsuKAv5b'}
page_content='
    Event: NAICE 2026
    Short Description: 2026 Nigeria Annual Intl Conference and Exhibition
    Description: Theme: Thri

Lets chunk the data

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Lets create our vector and vector db

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
print(embeddings)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")


Lets start retrieving

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant that helps users get information about a community.
You are chatting with a user about a community.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""


Lets setup chat function

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

Lets wire up Gradio

In [ ]:
gr.ChatInterface(answer_question).launch()